In [11]:
import numpy as np
import pandas as pd
import os

# General Setting:
pd.set_option('display.max_columns', None)

# Job Folder:
BASE_DIR = os.getcwd()

# Save Job:
OUTPUT_DIR = os.path.join(BASE_DIR, "data_processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Working directory:", BASE_DIR)
print("Output directory:", OUTPUT_DIR)

Working directory: /home/0580c1b6-c102-4fa8-b525-53f337aaf2e8
Output directory: /home/0580c1b6-c102-4fa8-b525-53f337aaf2e8/data_processed


In [12]:
# Load Data:

DATA_PATH = "DeltaT.xlsx"

df_raw = pd.read_excel(DATA_PATH)

print("Data loaded successfully")
print("Shape:", df_raw.shape)

Data loaded successfully
Shape: (24, 15)


In [13]:
#Inspecction:

print("\nColumns:")
print(df_raw.columns.tolist())

print("\nFirst rows:")
display(df_raw.head())

print("\nMissing values:")
print(df_raw.isna().sum())


Columns:
['Well', 'Lat', 'Long', 'ΔT 3km (°C)', 'ΔT 4km (°C)', 'ΔT 5km (°C)', 'ΔT 6km (°C)', 'Cond', 'Grad', 'Hflow', 'Tsup', 'Cond3', 'Cond4', 'Cond5', 'Cond6']

First rows:


,Well,Lat,Long,ΔT 3km (°C),ΔT 4km (°C),ΔT 5km (°C),ΔT 6km (°C),Cond,Grad,Hflow,Tsup,Cond3,Cond4,Cond5,Cond6
0,Altos del Arapey,-30.942,-57.518,84,111,137,163,2.7,28.8,78,19.5,3.449467,3.53710,3.58968,3.624733
1,Arapey,-30.949,-57.518,100,132,164,195,3.0,34.1,102,19.5,3.252200,3.38915,3.47132,3.526100
2,Arapey 2,-30.947,-57.523,79,105,130,155,2.6,27.4,71,19.5,3.470000,3.55250,3.60200,3.635000
3,Belén,-30.831,-57.698,66,88,109,129,2.7,23.2,63,19.7,2.943467,3.15760,3.28608,3.371733
4,Club Remeros,-31.224,-57.580,67,89,110,130,2.2,23.4,51,19.5,3.315267,3.43645,3.50916,3.557633



Missing values:
Well           0
Lat            0
Long           0
ΔT 3km (°C)    0
ΔT 4km (°C)    0
ΔT 5km (°C)    0
ΔT 6km (°C)    0
Cond           0
Grad           0
Hflow          0
Tsup           0
Cond3          0
Cond4          0
Cond5          0
Cond6          0
dtype: int64


In [14]:
df = df_raw.copy()

depths = [3, 4, 5, 6]

rows = []

for _, row in df.iterrows():
    for d in depths:
        temp = row[f'ΔT {d}km (°C)']
        cond = row[f'Cond{d}']

        rows.append({
        'Well': row['Well'],
        'Lat': row['Lat'],
        'Long': row['Long'],
        'Depth_km': d,
            'Temp_obs': temp,
            'Cond': cond,
            'Grad': row['Grad'],
            'Hflow': row['Hflow'],
            'Tsup': row['Tsup']
        })

df_long = pd.DataFrame(rows)

print("New shape:", df_long.shape)
display(df_long.head())

New shape: (96, 9)


,Well,Lat,Long,Depth_km,Temp_obs,Cond,Grad,Hflow,Tsup
0,Altos del Arapey,-30.942,-57.518,3,84,3.449467,28.8,78,19.5
1,Altos del Arapey,-30.942,-57.518,4,111,3.537100,28.8,78,19.5
2,Altos del Arapey,-30.942,-57.518,5,137,3.589680,28.8,78,19.5
3,Altos del Arapey,-30.942,-57.518,6,163,3.624733,28.8,78,19.5
4,Arapey,-30.949,-57.518,3,100,3.252200,34.1,102,19.5


In [15]:
# Basic Cleaning:

df = df_long.copy()

df = df.drop_duplicates()
df = df.reset_index(drop=True)

print("After cleaning:", df.shape)
df.head()

After cleaning: (96, 9)


,Well,Lat,Long,Depth_km,Temp_obs,Cond,Grad,Hflow,Tsup
0,Altos del Arapey,-30.942,-57.518,3,84,3.449467,28.8,78,19.5
1,Altos del Arapey,-30.942,-57.518,4,111,3.537100,28.8,78,19.5
2,Altos del Arapey,-30.942,-57.518,5,137,3.589680,28.8,78,19.5
3,Altos del Arapey,-30.942,-57.518,6,163,3.624733,28.8,78,19.5
4,Arapey,-30.949,-57.518,3,100,3.252200,34.1,102,19.5


In [16]:
# Columns:

feature_cols = ['Lat', 'Long', 'Depth_km', 'Temp_obs', 'Cond']
target_col = 'Hflow'

df_ml = df[feature_cols + [target_col]].copy()

In [17]:
df_ml = df_ml.dropna().reset_index(drop=True)
print("Final dataset shape:", df_ml.shape)

Final dataset shape: (96, 6)


In [18]:
# Validation:

def validate_dataframe(df, feature_cols, target_col):
    missing = [col for col in feature_cols + [target_col] if col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    if df.empty:
        raise ValueError("Dataframe is empty")

    print("n Dataframe validated successfully")
    print("Shape:", df.shape)

validate_dataframe(df_ml, feature_cols, target_col)

n Dataframe validated successfully
Shape: (96, 6)


In [19]:
display(df_ml.describe())

,Lat,Long,Depth_km,Temp_obs,Cond,Hflow
count,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000
mean,-31.112330,-57.280693,4.500000,104.604167,3.480742,67.833333
std,0.556214,0.679715,1.123903,39.073070,0.209200,22.225006
min,-32.721850,-57.962000,3.000000,37.000000,2.924767,43.000000
25%,-31.458000,-57.823250,3.750000,74.000000,3.362007,52.750000
50%,-31.086500,-57.551500,4.500000,101.000000,3.512130,62.500000
75%,-30.610750,-56.666427,5.250000,130.500000,3.604681,71.250000
max,-30.295000,-55.583810,6.000000,214.000000,3.784600,123.000000


In [20]:
# Export Data

output_path = os.path.join(OUTPUT_DIR, "df_ml_clean.csv")

df_ml.to_csv(output_path, index=False)

print("\nDataset saved at:")
print(output_path)


Dataset saved at:
/home/0580c1b6-c102-4fa8-b525-53f337aaf2e8/data_processed/df_ml_clean.csv
